[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Connecting and Executing


## What you will be able to do

Tell a connection and a cursor apart, and say which of the two a `with` block is ending. Read rows
one at a time, in batches, all at once and by iterating, and say what each one costs. Get rows back
as dictionaries, named tuples, single values or your own class, rather than as tuples you index by
position. Say what the `with` block around a connection commits, and what it does that psycopg 2 did
not. Ask which schema a connection can see. And do all of the same things in asyncpg, whose four
methods replace the cursor entirely.


## The idea

### The problem

A tuple is a fine thing to get back from a database until the query has six columns and somebody
inserts a seventh in the middle. `row[3]` was the price, and now it is the tax, and nothing raises:
the report is simply wrong from that deploy onward.

psycopg hands you tuples because the DB-API says so, and it lets you ask for something else with one
argument. Knowing that argument exists is most of this notebook, and the other part is knowing which
object to pass it to, because a connection and a cursor are different things that both look like
"the database" from a distance.

### What the two objects are

A **connection** is one session with one database. It holds one transaction at a time, and closing
it ends that transaction.

A **cursor** is the thing a statement runs on and rows come back from. Cursors are cheap, several
can exist on one connection, and closing one leaves the connection open and usable.

psycopg lets you skip the cursor for the common case: `conn.execute(...)` makes one, uses it, and
hands it back to you. That is a convenience rather than a different mechanism, and the cursor is
still there.

### Why it works that way

The DB-API, which `sqlite3` also follows, was written when every database library invented its own
names, and it settled on connections, cursors, `execute` and `fetch`. psycopg follows it, so most of
this notebook is knowledge you already have from `sqlite3`, plus the parts PostgreSQL adds.

asyncpg does not follow it. There is no cursor in ordinary asyncpg code: the connection has four
methods that each answer a different shape of question, which is less machinery and one more thing
to learn if you are moving between the two.

### Where this shows up

Every query anybody writes. The row factory in particular is the difference between code that reads
like `row["kind"]` and code full of numbered indexes that nobody dares reorder.

### What this notebook covers

The connection and the cursor, and what closing each one does. The four ways to read rows. The row
factories, including one that builds your own class. What the `with` block commits. The schema a
connection can see. asyncpg's four methods beside psycopg's cursor. Then the four failures, which
are mostly about using an object after its block has ended.

Every query here is written out in full, with its values in the SQL. That is deliberate and it is
temporary: putting a value into a query is the subject of **Placeholders and Identifiers**, and
doing it by hand is a bad habit this guide is about to break on purpose.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg
from psycopg.rows import dict_row

with psycopg.connect("dbname=guide") as conn:
    row = conn.execute("SELECT id, kind FROM events ORDER BY id LIMIT 1").fetchone()
    print("a row, by default:", row)
    try:
        row["kind"]
    except TypeError as error:
        print("  row['kind'] ->", type(error).__name__ + ":", error)

    with conn.cursor(row_factory=dict_row) as cur:
        row = cur.execute("SELECT id, kind FROM events ORDER BY id LIMIT 1").fetchone()
    print("with dict_row:    ", row)
    print("  row['kind'] ->", row["kind"])
```

```
a row, by default: (1, 'view')
  row['kind'] -> TypeError: tuple indices must be integers or slices, not str
with dict_row:     {'id': 1, 'kind': 'view'}
  row['kind'] -> view
```

The same query twice, and the only difference is one argument. The first row is a tuple, so asking
it for a column by name is a Python error rather than a database one. The second is a dictionary,
because the cursor was told what to build.


## Setup

Ten imports, both drivers, a server and the `events` table. This is the same cell as every other
notebook in this guide: it is slow the first time and quick afterwards.

- `psycopg` and `asyncpg` are the drivers
- `class_row`, `dict_row`, `namedtuple_row` and `scalar_row`, from `psycopg.rows`, are this
  notebook's subject
- `dataclass` is the class one of those factories builds
- `subprocess`, `sys`, `os`, `getpass` and `time` stand the server up, which **A Server of Your Own**
  takes apart
- `version` and `PackageNotFoundError` install the drivers where they are missing


In [1]:
import getpass
import os
import subprocess
import sys
import time
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg.rows import class_row, dict_row, namedtuple_row, scalar_row

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")


print("server:", start_server())
print(report())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows


## Worked examples

### A connection, and a cursor on it

Both have a `with` block, and the blocks do different things:


In [2]:
conn = psycopg.connect("dbname=guide")

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM events")
    print("inside the cursor block:", cur.fetchone())

print("the cursor is closed:    ", cur.closed)
print("the connection is not:   ", conn.closed)
print("and still answers:       ", conn.execute("SELECT count(*) FROM events").fetchone())
conn.close()


inside the cursor block: (5000,)
the cursor is closed:     True
the connection is not:    False
and still answers:        (5000,)


That is the distinction worth keeping. Ending a cursor's block closes the cursor and nothing else.
Ending a connection's block closes the connection, and every cursor on it with it.

`conn.execute(...)` on the line above never mentioned a cursor, and it made one anyway:


In [3]:
with psycopg.connect("dbname=guide") as conn:
    returned = conn.execute("SELECT 1")
    print("conn.execute gives back:", type(returned).__name__)

    with conn.cursor() as cur:
        print("cur.execute gives back: ", type(cur.execute("SELECT 1")).__name__,
              "which is the same cursor:", cur.execute("SELECT 1") is cur)


conn.execute gives back: Cursor
cur.execute gives back:  Cursor which is the same cursor: True


### Four ways to read

The rows are on the cursor once `execute` has run, and these are the ways to take them off it:


In [4]:
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.execute("SELECT id, kind FROM events ORDER BY id LIMIT 4")
    print("fetchone():  ", cur.fetchone())
    print("fetchmany(2):", cur.fetchmany(2))
    print("fetchall():  ", cur.fetchall(), "<- only what was left")

    cur.execute("SELECT id FROM events ORDER BY id LIMIT 3")
    print("iterating:   ", [row for row in cur])


fetchone():   (1, 'view')
fetchmany(2): [(2, 'purchase'), (3, 'click')]
fetchall():   [(4, 'view')] <- only what was left
iterating:    [(1,), (2,), (3,)]


Each call takes rows off the front, which is why `fetchall` at the end returned one row rather than
four. Iterating is the same thing in a loop and is the one to reach for when you are going to look at
every row once.

None of those four saves any memory over the others, which is the surprise in **Server-Side
Cursors**: by the time `execute` returns, the rows are already out of the server and in your process.
`rowcount` knows how many there are before you have fetched any of them:


In [5]:
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.execute("SELECT id, kind, payload FROM events WHERE kind = 'purchase'")
    print("rowcount before fetching anything:", cur.rowcount)
    for column in cur.description:
        print(f"  {column.name:<9} {column.type_display}")


rowcount before fetching anything: 1667
  id        int8
  kind      text
  payload   jsonb


`description` is how you find out what came back without knowing the query, which is what a row
factory uses to build anything other than a tuple.

### What a row is

Four factories, one query, four shapes:


In [6]:
query = "SELECT id, kind FROM events ORDER BY id LIMIT 1"

with psycopg.connect("dbname=guide") as conn:
    for name, factory in (("(none)", None), ("dict_row", dict_row),
                          ("namedtuple_row", namedtuple_row), ("scalar_row", scalar_row)):
        made = conn.cursor(row_factory=factory) if factory else conn.cursor()
        with made as cur:
            row = cur.execute(query).fetchone()
        print(f"  {name:<16} {type(row).__name__:<8} {row}")


  (none)           tuple    (1, 'view')
  dict_row         dict     {'id': 1, 'kind': 'view'}
  namedtuple_row   Row      Row(id=1, kind='view')
  scalar_row       int      1


`scalar_row` gives the first column rather than a container, which is what you want for a count or an
id and nothing else. `namedtuple_row` reads well and is still a tuple, so old code that indexes by
position keeps working.

The last one builds your class:


In [7]:
@dataclass
class Event:
    id: int
    kind: str


with psycopg.connect("dbname=guide") as conn:
    with conn.cursor(row_factory=class_row(Event)) as cur:
        event = cur.execute(query).fetchone()

print(event)
print("it is a real one:", isinstance(event, Event), "| event.kind:", event.kind)


Event(id=1, kind='view')
it is a real one: True | event.kind: view


The column names have to match the argument names, which is the whole contract. A row factory can be
set on the connection instead, and then every cursor made from it uses that factory:


In [8]:
with psycopg.connect("dbname=guide", row_factory=dict_row) as conn:
    print("set on the connection:", conn.execute(query).fetchone())


set on the connection: {'id': 1, 'kind': 'view'}


### What the with block commits

Nothing in this notebook has written anything yet. When it does, the block is what decides whether
the write survives:


In [9]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("DROP TABLE IF EXISTS notes")
    conn.execute("CREATE TABLE notes (id int, body text)")


def rows():
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT count(*) FROM notes").fetchone()[0]


print("to start with:", rows())

with psycopg.connect("dbname=guide") as conn:                       # no commit written anywhere
    conn.execute("INSERT INTO notes VALUES (1, 'the block ended normally')")
print("after a block that ended normally:", rows())

try:
    with psycopg.connect("dbname=guide") as conn:
        conn.execute("INSERT INTO notes VALUES (2, 'an error left the block')")
        raise RuntimeError("something went wrong after the write")
except RuntimeError:
    pass
print("after a block an exception left: ", rows())


to start with: 0
after a block that ended normally: 1
after a block an exception left:  1


The block commits when it ends normally and rolls back when an exception leaves it, which is the same
rule `db.atomic` follows in the **Peewee, Deep Dive** guide and the same one every transaction in
**Transactions and Errors** follows.

One thing to know if you are coming from psycopg 2, and then this guide will not mention it again:
there, the `with` block ended the transaction and left the connection open, so people wrote
`with conn:` in a loop. In psycopg 3 the block closes the connection:


In [10]:
with psycopg.connect("dbname=guide") as conn:
    pass

print("after the block, conn.closed:", conn.closed)
print("to keep the connection and end only the transaction, the block to use is conn.transaction()")


after the block, conn.closed: True
to keep the connection and end only the transaction, the block to use is conn.transaction()


### Which schema a connection can see

A table name with no schema in front of it is looked up along the `search_path`, which is a setting
on the session rather than anything in the query:


In [11]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS app")
    conn.execute("DROP TABLE IF EXISTS app.secrets")
    conn.execute("CREATE TABLE app.secrets (id int)")
    conn.execute("INSERT INTO app.secrets VALUES (1), (2)")

for options in ("-c search_path=app,public", "-c search_path=public"):
    with psycopg.connect("dbname=guide", options=options) as conn:
        found = conn.execute("SELECT to_regclass('secrets')").fetchone()[0]
        print(f"  {options:<28} search_path={conn.execute('SHOW search_path').fetchone()[0]:<12}"
              f" secrets resolves to: {found}")


  -c search_path=app,public    search_path=app,public   secrets resolves to: secrets
  -c search_path=public        search_path=public       secrets resolves to: None


The same query, the same database, two different answers about whether a table exists. `options` is
passed to the server at connection time, which is why it is a connection argument and not something
you can put in the connection string.

`to_regclass` is used above because it returns `None` rather than raising for a name that resolves to
nothing. Selecting from the table itself is the last of the Common errors.

### The same questions in asyncpg

asyncpg has no cursor in ordinary use. The connection has four methods, and which one you call is how
you say what shape you want back:


In [12]:
conn = await asyncpg.connect(database="guide")

print("fetch:    ", type(await conn.fetch("SELECT id FROM events ORDER BY id LIMIT 2")).__name__,
      await conn.fetch("SELECT id FROM events ORDER BY id LIMIT 2"))
print("fetchrow: ", await conn.fetchrow("SELECT id, kind FROM events ORDER BY id LIMIT 1"))
print("fetchval: ", await conn.fetchval("SELECT count(*) FROM events"))
print("execute:  ", repr(await conn.execute("SELECT 1")), "<- the command tag, not rows")


fetch:     list [<Record id=1>, <Record id=2>]
fetchrow:  <Record id=1 kind='view'>
fetchval:  5000
execute:   'SELECT 1' <- the command tag, not rows


A `Record` is asyncpg's row, and it does not need a factory to be readable: it indexes by position
like a tuple and by name like a dictionary, and `dict()` turns it into one.


In [13]:
row = await conn.fetchrow("SELECT id, kind FROM events ORDER BY id LIMIT 1")

print("by position:", row[0], "| by name:", row["kind"])
print("dict(row): ", dict(row))
print("keys:      ", list(row.keys()))
await conn.close()


by position: 1 | by name: view
dict(row):  {'id': 1, 'kind': 'view'}
keys:       ['id', 'kind']


### When to reach for which

| What you want | psycopg | asyncpg |
|---|---|---|
| one row | `cur.fetchone()` | `await conn.fetchrow(...)` |
| every row | `cur.fetchall()` | `await conn.fetch(...)` |
| one value | `cur.fetchone()[0]`, or `scalar_row` | `await conn.fetchval(...)` |
| a statement with no rows | `cur.execute(...)` | `await conn.execute(...)` |
| rows one at a time | iterate the cursor | `async for` over a cursor in a transaction |
| rows as dictionaries | `row_factory=dict_row` | already, a `Record` |
| rows as your own class | `row_factory=class_row(Event)` | build it from the `Record` |
| several statements, one session | one connection, cursors as needed | one connection |

The default in psycopg is `dict_row` for anything a person reads and `scalar_row` for a count. Leave
the tuples to code that immediately unpacks them, where the position is right there on the same line.

### A small report, finished

Everything above, as the thing it is for: a function that asks a question and hands back rows
somebody can read, with the connection opened and closed around it.


In [14]:
def summary(kind):
    """How many events of one kind there are, and the most recent few, as dictionaries."""
    with psycopg.connect("dbname=guide", row_factory=dict_row) as conn:
        total = conn.execute(
            f"SELECT count(*) AS total FROM events WHERE kind = '{kind}'").fetchone()["total"]
        recent = conn.execute(
            f"SELECT id, kind FROM events WHERE kind = '{kind}' ORDER BY id DESC LIMIT 3").fetchall()
    return {"kind": kind, "total": total, "recent": recent}


for kind in ("purchase", "click"):
    found = summary(kind)
    print(f"  {found['kind']:<9} {found['total']:>4} events | most recent: "
          f"{[row['id'] for row in found['recent']]}")


  purchase  1667 events | most recent: [5000, 4997, 4994]
  click     1666 events | most recent: [4998, 4995, 4992]


Every value in that function is written into the SQL with an f-string, which works here and is the
thing **Placeholders and Identifiers** exists to stop. A `kind` that came from a person rather than
from this cell would make it a very different function.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `psycopg.connect(..., row_factory=dict_row)` | rows as dictionaries, set once | What a row is |
| the `with` block | a connection closed however the function ends | What the with block commits |
| `.fetchone()["total"]` | a named column rather than `row[0]` | What a row is |
| `.fetchall()` | every row, when there are three of them | Four ways to read |
| `conn.execute(...)` with no cursor | the cursor psycopg made for you | A connection, and a cursor |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/02-connecting-and-executing-solutions.ipynb).

**1.** Count the events of each kind and print the result as dictionaries rather than tuples.


In [15]:
# your code here


**2.** Open one connection, make two cursors on it, run a different query on each, and show the
connection is still open after both cursor blocks have ended.


In [16]:
# your code here


**3.** Read the first four events with `fetchone`, then `fetchmany(2)`, then `fetchall`, and explain
the number of rows each one gave back.


In [17]:
# your code here


**4.** Define a dataclass for an event and get rows back as instances of it.


In [18]:
# your code here


**5.** Write a row inside a `with` block that raises afterwards, and show the row is not there.


In [19]:
# your code here


**6.** Ask the same question through asyncpg with `fetch`, `fetchrow` and `fetchval`, and print what
each one gave back.


In [20]:
# your code here


## Common errors

### psycopg.OperationalError: the connection is closed


In [21]:
with psycopg.connect("dbname=guide") as conn:
    print("inside the block:", conn.execute("SELECT count(*) FROM events").fetchone())

conn.execute("SELECT count(*) FROM events")


inside the block: (5000,)


OperationalError: the connection is closed

In psycopg 3 the `with` block closes the connection, so anything holding on to it afterwards is
holding a closed object. This is the one that catches people who keep a module-level `conn` and wrap
a function body in `with conn:` out of habit from psycopg 2.

Open the connection where it is used, or keep it and end only the transaction:


In [22]:
keeper = psycopg.connect("dbname=guide")

with keeper.transaction():                                          # the transaction, not the connection
    keeper.execute("SELECT 1")
print("after a transaction block, closed:", keeper.closed)

with keeper.transaction():
    keeper.execute("SELECT 2")
print("and it can be used again:", keeper.execute("SELECT count(*) FROM events").fetchone())
keeper.close()


after a transaction block, closed: False
and it can be used again: (5000,)


### psycopg.InterfaceError: the cursor is closed


In [23]:
conn = psycopg.connect("dbname=guide")
with conn.cursor() as cur:
    cur.execute("SELECT 1")

cur.execute("SELECT 2")


InterfaceError: the cursor is closed

A different class from the one above, for a different object. `InterfaceError` is the DB-API's word
for a problem with the driver's own objects rather than with the database, and the database was never
asked anything here.

The connection is untouched, which is the point of the pair:


In [24]:
print("the connection is still open:", not conn.closed)
with conn.cursor() as fresh:
    print("a new cursor works:", fresh.execute("SELECT count(*) FROM events").fetchone())
conn.close()


the connection is still open: True
a new cursor works: (5000,)


### TypeError: tuple indices must be integers or slices, not str


In [25]:
with psycopg.connect("dbname=guide") as conn:
    row = conn.execute("SELECT id, kind FROM events ORDER BY id LIMIT 1").fetchone()

row["kind"]


TypeError: tuple indices must be integers or slices, not str

Not a database error at all. The query worked, the row came back, and a tuple was asked for a string
index. It is in this list because of where it sends you looking: the SQL is fine and the problem is
one argument on the cursor.

The fix is the row factory, and it is worth setting on the connection so that nobody has to remember
it per cursor:


In [26]:
with psycopg.connect("dbname=guide", row_factory=dict_row) as conn:
    row = conn.execute("SELECT id, kind FROM events ORDER BY id LIMIT 1").fetchone()
print(row, "| row['kind'] ->", row["kind"])


{'id': 1, 'kind': 'view'} | row['kind'] -> view


### psycopg.ProgrammingError: the last operation didn't produce records


In [27]:
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.execute("CREATE TEMP TABLE scratch (id int)")
    cur.fetchall()


ProgrammingError: the last operation didn't produce records (command status: CREATE TABLE)

`CREATE TABLE` produces no rows, so there is nothing to fetch and the cursor says so rather than
handing back an empty list. The message names the command it ran, which is usually enough to see
what happened.

An empty list and no rows at all are different things, and this is the line between them. A `SELECT`
that matched nothing produces a result set with no rows in it, and fetching from that is fine:


In [28]:
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    cur.execute("SELECT id FROM events WHERE kind = 'nothing is this kind'")
    print("a SELECT that matched nothing:", cur.fetchall(), "| rowcount:", cur.rowcount)

    cur.execute("CREATE TEMP TABLE scratch (id int)")               # this session only
    cur.execute("INSERT INTO scratch VALUES (1) RETURNING id")      # a write can produce rows
    print("an INSERT with RETURNING:     ", cur.fetchall())


a SELECT that matched nothing: [] | rowcount: 0
an INSERT with RETURNING:      [(1,)]


### psycopg.errors.UndefinedTable: relation "secrets" does not exist


In [29]:
with psycopg.connect("dbname=guide", options="-c search_path=public") as conn:
    conn.execute("SELECT count(*) FROM secrets")


UndefinedTable: relation "secrets" does not exist
LINE 1: SELECT count(*) FROM secrets
                             ^

The table is in the `app` schema and this connection was told to look only in `public`. The same
statement on a connection with `app` in its `search_path` works, which is what the worked example
showed.

Two ways to be sure, and the second one is the one to prefer in anything that is not a script:


In [30]:
with psycopg.connect("dbname=guide", options="-c search_path=app,public") as conn:
    print("with app on the search_path:", conn.execute("SELECT count(*) FROM secrets").fetchone())

with psycopg.connect("dbname=guide") as conn:
    print("or name the schema outright: ", conn.execute("SELECT count(*) FROM app.secrets").fetchone())


with app on the search_path: (2,)
or name the schema outright:  (2,)


## Recap

- A connection is a session with one database and holds one transaction. A cursor is what a statement
  runs on. Closing a cursor leaves the connection open; closing the connection closes everything.
- `conn.execute(...)` makes a cursor for you and gives it back, which is the short form of the same
  thing.
- `fetchone`, `fetchmany`, `fetchall` and iterating all take rows off the front of the same result,
  and none of them saves memory over the others.
- `rowcount` and `description` answer how many rows and which columns before you fetch anything.
- A row is a tuple unless the cursor was given a `row_factory`. `dict_row`, `namedtuple_row`,
  `scalar_row` and `class_row(YourClass)` are the four, and the factory can be set on the connection.
- A connection's `with` block commits when it ends normally, rolls back when an exception leaves it,
  and in psycopg 3 closes the connection either way. `conn.transaction()` is the block that does not.
- `options="-c search_path=..."` decides which schema an unqualified table name is looked up in.
- asyncpg has no cursor in ordinary use: `fetch`, `fetchrow`, `fetchval` and `execute` are the four
  shapes, and its `Record` indexes by position and by name without any factory.


## What is next

The **Transactions and Errors** notebook is what happens when a statement fails: the one that poisons
every statement after it until the transaction ends, the SQLSTATE code that says which constraint
was violated, and the savepoint that costs you one row instead of the whole batch.


---

&#8592; **Previous:** [A Server of Your Own](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/01-a-server-of-your-own.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Transactions and Errors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/03-transactions-and-errors.ipynb) &#8594;
